## Comparación entre modelos de presición y velocidad

El test hace uso de las imágenes ubicadas en la carpeta test_img, empleando el código existen en el proyecto.

In [1]:
import sys
import os

# Añadir la carpeta raíz del proyecto al sys.path
project_root = os.path.abspath(os.path.join(os.path.dirname(os.getcwd() + "/test"), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import cv2
from app.embedings import crear_embedding
from db.model_user import guardar_usuario
from app.verificacion_faiss import construir_indice, buscar_usuario_por_embedding

c:\Users\Sam\Desktop\project_ia_face\faceenv\lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.7' (you have '2.0.6'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\Sam/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'],

In [ ]:
# 1. Cargar imágenes de la carpeta img_test
imagenes = []


imagenes.append(cv2.imread("../img_test/compare/ana1.jpg"))
imagenes.append(cv2.imread("../img_test/compare/ana2.jpg"))
imagenes.append(cv2.imread("../img_test/compare/ana3.jpg"))
imagenes.append(cv2.imread("../img_test/compare/ana4.jpg"))

# 2. Crear embeddings
embeddings = []
for img in imagenes:
    try:
        emb = crear_embedding(img)
        embeddings.append(emb)
    except Exception as e:
        print(f"Error procesando imagen: {e}")


# 3. Guardar usuario de prueba
if embeddings:
    guardar_usuario(
        nombre="Ana test",
        codigo="TST001",
        facultad="Ingeniería",
        carrera="Chistemas",
        embeddings=embeddings
    )
    print("Usuario y embeddings guardados.")

In [3]:
# 4. Prueba de verificación con FAISS

#crearemos un nuevo embedding a partir de la 5ta imagen
nueva_imagen = cv2.imread("../img_test/compare/ana5.jpg")
nuevo_embedding = crear_embedding(nueva_imagen)

if embeddings:
    print("\n--- Prueba modo velocidad ---")
    construir_indice(modo='velocidad')
    usuario, sim = buscar_usuario_por_embedding(nuevo_embedding)
    print("Este primer resultado emplea el embedding promedio de los 4 embeddings guardados por usuario")
    print(f"Resultado velocidad: {usuario['nombre'] if usuario else 'No encontrado'} (similitud: {sim:.3f})")


    print("\n--- Prueba modo presicion ---")
    construir_indice(modo='presicion')
    usuario, sim = buscar_usuario_por_embedding(nuevo_embedding)
    print("Este segundo resultado emplea los 4 embeddings guardados por usuario")
    print(f"Resultado presicion: {usuario['nombre'] if usuario else 'No encontrado'} (similitud: {sim:.3f})")
    print("Indice de embedding seleccionado: ", usuario['idEmb'])


--- Prueba modo velocidad ---
Este primer resultado emplea el embedding promedio de los 4 embeddings guardados por usuario
Resultado velocidad: Ana test (similitud: 0.660)

--- Prueba modo presicion ---
Este segundo resultado emplea los 4 embeddings guardados por usuario
Resultado presicion: Ana test (similitud: 0.615)
Indice de embedding seleccionado:  12
